# 07 · Calidad de la etiqueta y severidad

El modelo diario (cuaderno 05) se entrena con «un día con huaico catalogado». Pero no todas las fechas del catálogo valen lo mismo, y el catálogo no describe igual todas las épocas. Este cuaderno pregunta cuánto cambia la habilidad del modelo con la calidad de la etiqueta.

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Mide qué tan bien el tiempo explica las fechas que alguien registró. Ante una emergencia llama al 115 (INDECI).

## Teorías (escritas antes de mirar un solo puntaje)

- **T12** El modelo predice mejor los días mejor documentados. Etiquetas alternativas: **A** fechas en 2 o más archivos del catálogo o con confianza alta; **B** días con 2 o más quebradas ubicadas activas; **C** días citados por 3 o más tipos de fuente; **D** base ampliada con el día anterior. Se espera mayor *lift* (AP / tasa base) y mayor AUC en A, B y C que en la base.
- **T13** El sesgo de reporte cambia por era (1984-1998, 1999-2008, 2009-2026): la habilidad y la tasa de fechas por temporada cambian con cuántas fuentes existen por año. Se espera menos fechas por día de lluvia extrema en 1999-2008 y control con una variable de era.
- **T14** El catálogo mezcla inundaciones con huaicos; quitar las fechas que solo hablan de inundación cambia poco la habilidad (hay pocas), pero quitar las que mencionan inundación en cualquier fila la baja si esas son las más grandes.
- **T15** Los días con la probabilidad más alta sin fecha catalogada son en parte huaicos sin registrar. Se buscan en la web; solo se anota lo que se encuentra con fuente.

Validación: dejando fuera una temporada cada vez; intervalos de 90 % por bootstrap sobre temporadas. La referencia es la climatología por mes.

In [ ]:
import dataclasses
from pathlib import Path

import numpy as np
import pandas as pd

from illapa_research import etiquetas_calidad as ec, filtro, modelo_diario as md, quebradas as q
from illapa_research.catalogo import CATALOGO, cargar_cuencas, cargar_todos

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
D = md.cargar_datos(RAIZ)
print(D.inventario["fuente_era5"])
print("mar:", D.inventario["cajas_sst"], "| último día:", D.inventario["ultimo_dia"])
mask = D.inventario["mask"]

## 1. Corroboración y severidad de cada fecha

Una fila por fecha con día exacto y confianza por encima de `baja`: en cuántos archivos del catálogo aparece, cuántos *tipos* de fuente la citan (instituciones, ciencia, DesInventar, prensa de Lima, televisión), si es de confianza alta, si todas sus filas hablan solo de inundación y cuántas quebradas ubicadas se activaron ese día.

In [ ]:
filas = ec.leer_filas(RAIZ / "catalogo")
cuencas = filtro.cuencas_a_tabla(cargar_cuencas(RAIZ / "catalogo" / "cuencas.geojson"))
refs = q.cargar_referencias(RAIZ / "catalogo" / "quebradas_referencias.csv")
hist = filtro.asignar_historial(cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True), refs, cuencas)
ubicadas = set(cuencas.loc[cuencas.ubicacion == "referencia", "slug"])
T = ec.agregar_severidad(ec.tabla_fechas(filas), hist.activaciones, ubicadas)
L = ec.etiquetas_alternativas(T, D.F.index)
print(len(T), "fechas")
pd.DataFrame({"fechas": {k: int(v.sum()) for k, v in L.items()},
              "en temporada de lluvias 1984+": {k: int(v[mask].sum()) for k, v in L.items()}})

In [ ]:
T[["n_archivos", "n_tipos_fuente", "n_quebradas"]].describe().round(2)

## 2. T12 · Probabilidad fuera de muestra y etiquetas alternativas

Una sola probabilidad por día (modelo `lluvia+antecedentes`, entrenado con la etiqueta base, sin ver su temporada). Se mide contra cada etiqueta. *Lift* = AP / tasa base de esa etiqueta. El Brier skill de las etiquetas más raras sale negativo porque la probabilidad está calibrada a la tasa de la base: úsense el *lift* y el AUC.

In [ ]:
res = md.correr_teorias(D, "temporada", n_jobs=4, teorias=("lluvia+antecedentes",), con_oraculo=False)
X, y, ts, mes = md.matrices(D)
p, pref = res["lluvia+antecedentes"].p, res["fecha"].p
cols = ["eventos", "ap", "ap_p5", "ap_p95", "lift", "lift_p5", "lift_p95", "auc", "aciertos_con_5pc_alarmas"]
ev = {k: ec.metricas_sobre(v[mask].to_numpy(), p, pref, ts, n=200) for k, v in L.items()}
pd.DataFrame(ev).T[cols].round(3)

**Reentrenando** con cada etiqueta (el modelo aprende la etiqueta alternativa; las demás columnas no cambian):

In [ ]:
rt = {}
for k in ("A_corroborada", "B_severa", "D_ampliada", "E_sin_inundacion"):
    r = md.correr_teorias(dataclasses.replace(D, y=L[k]), "temporada", n_jobs=4,
                          teorias=("lluvia+antecedentes",), con_oraculo=False)
    rt[k] = ec.metricas_sobre(L[k][mask].to_numpy(), r["lluvia+antecedentes"].p, r["fecha"].p, ts, n=200)
pd.DataFrame(rt).T[cols].round(3)

## 3. T13 · Era, fuentes y detección

Habilidad por era con la misma probabilidad fuera de muestra; fuentes que aportan por temporada; y un índice de detección: fechas catalogadas por cada día de lluvia extrema (percentil 99 de la lluvia diaria de Chosica, mismo umbral para todos los años).

In [ ]:
era = ec.era_de(ts)
yb = L["base"][mask].to_numpy()
por_era = {e: ec.metricas_sobre(yb, p, pref, ts, n=200, mascara=era == e) for e in ("1984-1998", "1999-2008", "2009-2026")}
pd.DataFrame(por_era).T[["eventos", "dias", "ap", "ap_p5", "ap_p95", "lift", "auc", "aciertos_con_5pc_alarmas"]].round(3)

In [ ]:
fp = ec.fuentes_por_temporada(filas)
di = ec.indice_deteccion(D.F, D.y)
resumen = fp.groupby(ec.era_de(fp.index.to_numpy())).mean().round(2)
suma = di.groupby(ec.era_de(di.index.to_numpy()))[["dias_extremos", "fechas"]].sum()
resumen["fechas_por_dia_extremo"] = (suma.fechas / suma.dias_extremos).round(2)
resumen

**Controles.** (a) Variables ficticias de era como columnas; (b) recalibrar la probabilidad con la tasa de cada era (esto usa las etiquetas de la era, así que es un techo, no un modelo).

In [ ]:
Xe = X["lluvia+antecedentes"].copy()
for e in ("1999-2008", "2009-2026"):
    Xe["era_" + e] = (era == e).astype(float)
folds = [f for f in md.E.pliegues(ts, "temporada") if len(f)]
r_era = md.validar_anidado(Xe, y, ts, folds, n_jobs=4)
pr = p.copy()
for e in np.unique(era):
    m = era == e
    pr[m] = np.clip(p[m] * yb[m].mean() / p[m].mean(), 1e-6, 1)
ctrl = {"sin control": ec.metricas_sobre(yb, p, pref, ts, n=100),
        "con ficticias de era": ec.metricas_sobre(yb, r_era.p, pref, ts, n=100),
        "recalibrada por era (techo)": ec.metricas_sobre(yb, pr, pref, ts, n=100)}
pd.DataFrame(ctrl).T[["ap", "lift", "auc", "brier_skill", "aciertos_con_5pc_alarmas"]].round(3)

## 4. T14 · Inundaciones mezcladas con huaicos

`solo_inundacion`: todas las filas de la fecha hablan de inundación o desborde, nunca de huaico, y no nombran una quebrada. `menciona_inundacion`: alguna fila usa esas palabras (aunque también hable de huaico, como el 22 de marzo de 1983).

In [ ]:
idx_m = D.F.index[mask]
out = {}
for nombre, col in (("solo inundación", "solo_inundacion"), ("menciona inundación", "menciona_inundacion")):
    fechas = set(T.index[T[col]])
    fuera = np.array([d in fechas for d in idx_m])
    out[f"sin las {nombre} ({int(fuera.sum())} filas)"] = ec.metricas_sobre(yb, p, pref, ts, n=200, mascara=~fuera)
    out[f"solo las {nombre} + días sin evento"] = ec.metricas_sobre(yb, p, pref, ts, n=200, mascara=fuera | (yb == 0))
out["base"] = ev["base"]
pd.DataFrame(out).T[["eventos", "ap", "lift", "auc", "aciertos_con_5pc_alarmas"]].round(3)

## 5. T15 · Los mayores falsos positivos fuera de muestra

Los 20 días con mayor probabilidad que no tienen fecha catalogada ese día ni el anterior ni el siguiente (uno por racimo de 3 días). Cada uno es una pregunta para la hemeroteca: ¿huaico sin registrar o falsa alarma? **Aquí no se resuelve** lo que no tiene fuente.

In [ ]:
ps = pd.Series(p, index=idx_m)
ys = pd.Series(yb, index=idx_m)
fp20 = ec.falsos_positivos(ps, ys, 20)
fp20["lluvia_1d_chosica_mm"] = D.F.loc[fp20.index, "chosica_h_lluvia_1d"].round(1)
fp20["lluvia_3d_chosica_mm"] = D.F.loc[fp20.index, "chosica_h_lluvia_3d"].round(1)
fp20["temporada_nino"] = [bool(D.temporadas.nino.get(s, False)) for s in md.E.temporadas_de(fp20.index)]
fp20["p_solo_fecha"] = pd.Series(pref, index=idx_m).loc[fp20.index].round(3)
fp20.round(3)

Conclusiones, límites y qué falta están en `docs/07-calidad-de-etiquetas.md`, con la versión de los datos con que se corrió.